# 01. Data Exploration & Stratified Partitioning

**Course**: CS 328 — Introduction to Data Science  
**Project**: *Does Graph-Based Retrieval Beat Vector Search on Multi-Hop Questions?*  
**Phase 1**: Exploring 2WikiMultiHopQA dataset, checking question types, verifying disjoint splits, and inspecting symmetric passage representations.

In [ ]:
import sys
from pathlib import Path
# Add project root to sys.path
sys.path.insert(0, str(Path.cwd().parent))

from src import config
from src.data_loader import get_or_create_splits, normalize_text, normalize_answer
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the pre-registered disjoint splits
eval_split, tune_split = get_or_create_splits()
print(eval_split.summary())
print("
" + tune_split.summary())

## 1. Verify Zero Data Snooping (Disjoint Splits Invariant)

In [ ]:
overlap = eval_split.question_ids.intersection(tune_split.question_ids)
print(f"Question overlap count: {len(overlap)}")
assert len(overlap) == 0, "Splits must have zero overlap!"

## 2. Question Types Distribution (Stratification Analysis)

In [ ]:
eval_df = pd.DataFrame([{
    "id": q.id,
    "type": q.type,
    "question": q.question,
    "answer": q.answer,
    "gold_count": len(q.gold_titles),
    "context_count": len(q.context_titles)
} for q in eval_split.questions])

print("Evaluation Split Type Counts:")
print(eval_df["type"].value_counts())

# Plot question type distribution
plt.figure(figsize=(8, 4))
sns.countplot(data=eval_df, x="type", hue="type", order=config.QUESTION_TYPES, palette="viridis", legend=False)
plt.title("Evaluation Set (N=200) Stratified Question Types")
plt.xlabel("Question Type")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("../figures/question_types_distribution.png", dpi=150)
plt.show()

## 3. Sample Questions & Symmetric Passage Representation

In [ ]:
for qtype in config.QUESTION_TYPES:
    sample = eval_split.filter_by_type(qtype)[0]
    print(f"
=== [{qtype.upper()}] ===")
    print(f"Q: {sample.question}")
    print(f"A: {sample.answer}")
    print(f"Gold Titles: {sample.gold_titles}")
    first_gold = next(iter(sample.gold_titles))
    p = eval_split.get_passage(first_gold)
    if p:
        print(f"Passage Snippet:
{p.full_text[:180]}...")